# Notebook 01 — Data Collection

**Proyek:** Segmentasi Pelanggan untuk Mengatasi Ketidakefektifan Strategi Pemasaran Menggunakan Metode RFM dan Algoritma K-Means

## Tujuan Notebook
Tahap **Data Collection** bertujuan untuk:
1. Mengidentifikasi sumber data dan deskripsinya.
2. Memuat seluruh tabel dataset ke dalam memori.
3. Mendokumentasikan *schema* (kolom, tipe data, ukuran).
4. Memverifikasi *integritas referensial* antar-tabel (apakah primary–foreign key cocok).
5. Menghasilkan ringkasan koleksi data (data dictionary) yang akan dipakai di tahap berikutnya.

## 1. Sumber Data

**Dataset:** *Brazilian E-Commerce Public Dataset by Olist*
**Penerbit:** Olist Store (marketplace e-commerce terbesar di Brazil) – dipublikasikan di Kaggle (2018).
**Lisensi:** CC BY-NC-SA 4.0
**URL:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
**Periode:** September 2016 – Oktober 2018 (~ 2 tahun).
**Cakupan:** 99.441 order, 96.096 pelanggan unik, 32.951 produk, 3.095 seller.

Dataset terdiri dari **9 file CSV** yang saling berelasi (skema tipe *star schema*) dengan `order_id` dan `customer_id` sebagai kunci utama.

| File | Deskripsi singkat |
|------|-------------------|
| `olist_orders_dataset.csv` | Tabel induk order (timestamp, status). |
| `olist_order_items_dataset.csv` | Item per order, harga, freight. |
| `olist_order_payments_dataset.csv` | Pembayaran (tipe, cicilan, nilai). |
| `olist_order_reviews_dataset.csv` | Review pelanggan (skor 1–5). |
| `olist_customers_dataset.csv` | Pelanggan & lokasi. |
| `olist_products_dataset.csv` | Atribut produk. |
| `olist_sellers_dataset.csv` | Atribut seller. |
| `olist_geolocation_dataset.csv` | Geolokasi zip code. |
| `product_category_name_translation.csv` | Translasi kategori produk PT→EN. |

## 2. Setup & Import Library

In [9]:
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
REPORTS_DIR = PROJECT_ROOT / 'reports'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Raw data dir :', RAW_DIR)
print('Python ver.  :', sys.version.split()[0])
print('Pandas ver.  :', pd.__version__)

Project root : /Users/rafafawwaz/Documents/sem 6/PDM/Riset Data Mining - Rafa
Raw data dir : /Users/rafafawwaz/Documents/sem 6/PDM/Riset Data Mining - Rafa/data/raw
Python ver.  : 3.13.2
Pandas ver.  : 3.0.2


## 3. Inventaris File Mentah

In [10]:
files = sorted(RAW_DIR.glob('*.csv'))
inventory = []
for f in files:
    size_mb = f.stat().st_size / (1024**2)
    n_cols = pd.read_csv(f, nrows=0).shape[1]
    with open(f, 'rb') as fh:
        n_rows = sum(1 for _ in fh) - 1
    inventory.append({
        'file': f.name,
        'size_MB': round(size_mb, 2),
        'rows': n_rows,
        'cols': n_cols,
    })
inv_df = pd.DataFrame(inventory).sort_values('size_MB', ascending=False).reset_index(drop=True)
inv_df

,file,size_MB,rows,cols
0,olist_geolocation_dataset.csv,58.44,1000163,5
1,olist_orders_dataset.csv,16.84,99441,8
2,olist_order_items_dataset.csv,14.72,112650,7
3,olist_order_reviews_dataset.csv,13.78,104719,7
4,olist_customers_dataset.csv,8.62,99441,5
5,olist_order_payments_dataset.csv,5.51,103886,5
6,olist_products_dataset.csv,2.27,32951,9
7,olist_sellers_dataset.csv,0.17,3095,4
8,product_category_name_translation.csv,0.00,71,2


## 4. Loading Seluruh Tabel

In [11]:
def load_olist():
    """Memuat seluruh tabel Olist sebagai dictionary of DataFrame."""
    parse_dates = {
        'orders': ['order_purchase_timestamp', 'order_approved_at',
                   'order_delivered_carrier_date', 'order_delivered_customer_date',
                   'order_estimated_delivery_date'],
        'order_items': ['shipping_limit_date'],
        'order_reviews': ['review_creation_date', 'review_answer_timestamp'],
    }
    spec = {
        'orders'         : 'olist_orders_dataset.csv',
        'order_items'    : 'olist_order_items_dataset.csv',
        'order_payments' : 'olist_order_payments_dataset.csv',
        'order_reviews'  : 'olist_order_reviews_dataset.csv',
        'customers'      : 'olist_customers_dataset.csv',
        'products'       : 'olist_products_dataset.csv',
        'sellers'        : 'olist_sellers_dataset.csv',
        'geolocation'    : 'olist_geolocation_dataset.csv',
        'category_trans' : 'product_category_name_translation.csv',
    }
    data = {}
    for k, fname in spec.items():
        kwargs = {}
        if k in parse_dates:
            kwargs['parse_dates'] = parse_dates[k]
        df = pd.read_csv(RAW_DIR / fname, **kwargs)
        data[k] = df
        print(f'  {k:<15s} -> shape {df.shape}')
    return data

print('Memuat seluruh tabel Olist...')
data = load_olist()
print('\nTotal tabel termuat :', len(data))

Memuat seluruh tabel Olist...
  orders          -> shape (99441, 8)
  order_items     -> shape (112650, 7)
  order_payments  -> shape (103886, 5)
  order_reviews   -> shape (99224, 7)
  customers       -> shape (99441, 5)
  products        -> shape (32951, 9)
  sellers         -> shape (3095, 4)
  geolocation     -> shape (1000163, 5)
  category_trans  -> shape (71, 2)

Total tabel termuat : 9


## 5. Schema (Data Dictionary)

In [12]:
schema_rows = []
for tbl_name, df in data.items():
    for col in df.columns:
        schema_rows.append({
            'table'  : tbl_name,
            'column' : col,
            'dtype'  : str(df[col].dtype),
            'n_unique': df[col].nunique(dropna=True),
            'n_null' : int(df[col].isna().sum()),
            'pct_null': round(df[col].isna().mean()*100, 2),
            'sample' : df[col].dropna().astype(str).head(1).to_list()[0] if df[col].notna().any() else None,
        })
schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv(REPORTS_DIR / 'data_dictionary.csv', index=False)
print('Data dictionary tersimpan di reports/data_dictionary.csv')
schema_df.head(20)

Data dictionary tersimpan di reports/data_dictionary.csv


,table,column,dtype,n_unique,n_null,pct_null,sample
0,orders,order_id,str,99441,0,0.00,e481f51cbdc54678b7cc49136f2d6af7
1,orders,customer_id,str,99441,0,0.00,9ef432eb6251297304e76186b10a928d
2,orders,order_status,str,8,0,0.00,delivered
3,orders,order_purchase_timestamp,datetime64[us],98875,0,0.00,2017-10-02 10:56:33
4,orders,order_approved_at,datetime64[us],90733,160,0.16,2017-10-02 11:07:15
5,orders,order_delivered_carrier_date,datetime64[us],81018,1783,1.79,2017-10-04 19:55:00
6,orders,order_delivered_customer_date,datetime64[us],95664,2965,2.98,2017-10-10 21:25:13
7,orders,order_estimated_delivery_date,datetime64[us],459,0,0.00,2017-10-18
8,order_items,order_id,str,98666,0,0.00,00010242fe8c5a6d1ba2dd792cb16214
9,order_items,order_item_id,int64,21,0,0.00,1


## 6. Verifikasi Integritas Referensial

Memeriksa apakah *foreign key* di tabel detail benar-benar ada di tabel induk.

In [13]:
def coverage(child_df, child_col, parent_df, parent_col):
    child_set  = set(child_df[child_col].dropna().unique())
    parent_set = set(parent_df[parent_col].dropna().unique())
    matched = child_set & parent_set
    return {
        'child' : f'{child_col}',
        'parent': f'{parent_col}',
        'child_unique' : len(child_set),
        'parent_unique': len(parent_set),
        'matched'      : len(matched),
        'coverage_%'   : round(100 * len(matched) / max(len(child_set), 1), 2),
    }

ref_checks = [
    coverage(data['orders'],         'customer_id', data['customers'], 'customer_id'),
    coverage(data['order_items'],    'order_id',    data['orders'],    'order_id'),
    coverage(data['order_payments'], 'order_id',    data['orders'],    'order_id'),
    coverage(data['order_reviews'],  'order_id',    data['orders'],    'order_id'),
    coverage(data['order_items'],    'product_id',  data['products'],  'product_id'),
    coverage(data['order_items'],    'seller_id',   data['sellers'],   'seller_id'),
]
ref_df = pd.DataFrame(ref_checks)
ref_df

,child,parent,child_unique,parent_unique,matched,coverage_%
0,customer_id,customer_id,99441,99441,99441,100.0
1,order_id,order_id,98666,99441,98666,100.0
2,order_id,order_id,99440,99441,99440,100.0
3,order_id,order_id,98673,99441,98673,100.0
4,product_id,product_id,32951,32951,32951,100.0
5,seller_id,seller_id,3095,3095,3095,100.0


## 7. Sampling untuk Pemeriksaan Cepat

Beberapa baris pertama dari tabel-tabel utama yang akan digunakan dalam analisis RFM:
- `orders`     -> **Recency & Frequency** (timestamp, status)
- `customers`  -> Pemetaan ke `customer_unique_id`
- `order_payments` -> **Monetary** (nilai pembayaran)

In [14]:
print('--- ORDERS ---')
display(data['orders'].head(3))
print('\n--- CUSTOMERS ---')
display(data['customers'].head(3))
print('\n--- ORDER PAYMENTS ---')
display(data['order_payments'].head(3))

--- ORDERS ---


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04



--- CUSTOMERS ---


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



--- ORDER PAYMENTS ---


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


## 8. Snapshot Periode Data

In [15]:
orders = data['orders']
print('Periode order_purchase_timestamp :',
      orders['order_purchase_timestamp'].min(), '->',
      orders['order_purchase_timestamp'].max())
print('Total order  :', len(orders))
print('Total customer_id (per-order)   :', orders['customer_id'].nunique())
print('Total customer_unique_id (org.) :', data['customers']['customer_unique_id'].nunique())

Periode order_purchase_timestamp : 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
Total order  : 99441
Total customer_id (per-order)   : 99441
Total customer_unique_id (org.) : 96096


## 9. Simpan Hasil Tahap Data Collection

In [16]:
inv_df.to_csv(REPORTS_DIR / 'data_inventory.csv', index=False)
ref_df.to_csv(REPORTS_DIR / 'referential_integrity.csv', index=False)

collection_log = {
    'collected_at' : datetime.now().isoformat(timespec='seconds'),
    'source'       : 'Kaggle - Brazilian E-Commerce Public Dataset by Olist',
    'license'      : 'CC BY-NC-SA 4.0',
    'period_start' : str(orders['order_purchase_timestamp'].min()),
    'period_end'   : str(orders['order_purchase_timestamp'].max()),
    'n_tables'     : len(data),
    'total_rows'   : int(inv_df['rows'].sum()),
    'total_size_MB': float(inv_df['size_MB'].sum()),
}
with open(REPORTS_DIR / 'collection_log.json', 'w') as f:
    json.dump(collection_log, f, indent=2)
collection_log

{'collected_at': '2026-04-27T15:49:25',
 'source': 'Kaggle - Brazilian E-Commerce Public Dataset by Olist',
 'license': 'CC BY-NC-SA 4.0',
 'period_start': '2016-09-04 21:15:19',
 'period_end': '2018-10-17 17:30:18',
 'n_tables': 9,
 'total_rows': 1556417,
 'total_size_MB': 120.35}

## 10. Kesimpulan Tahap Data Collection

1. Seluruh **9 tabel** Olist berhasil dimuat tanpa error.
2. Tabel `orders` memiliki **99.441 baris** dengan periode **September 2016 – Oktober 2018**.
3. Integritas referensial **`customer_id`, `order_id`, `product_id`, `seller_id`** terhadap tabel induknya sangat baik (mendekati 100%).
4. *Data dictionary* dan *referential integrity report* telah disimpan ke folder `reports/` untuk dokumentasi.

Tahap berikutnya: **Notebook 02 – Exploratory Data Analysis (EDA)**.